# 🧠 M4 Meta-Memory Engine — Turnkey Benchmark Suite
**Paper**: *Meta-Memory for Large Language Models* (IEEE TASLPRO 2026)
**Evaluation Target**: LoCoMo Benchmark (Phase 1)

This turnkey notebook runs the complete 4-layer Meta-Memory architecture on **Google Colab** (with free T4 GPU) or **Kaggle Notebooks**:
1. **Layer 1: Online Meta-Learning** (Clue extraction & semantic disambiguation)
2. **Layer 2: Memory Chain Construction** (Dynamic graph topology & shared link construction)
3. **Layer 3: Self-Monitoring Meta-Updating** (Chain calibration & dormant node compression)
4. **Layer 4: Self-Reflective Retrieval** (Bidirectional chain expansion & reflection confidence)

---

### Step 0: Check GPU Acceleration
Verify NVIDIA GPU (Tesla T4, P100, V100, or A100) is enabled. If not, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi

### Step 1: Install Ollama Engine & Start Background Service

In [ ]:
import subprocess, time, urllib.request

# 1. Install Ollama binary
!command -v ollama >/dev/null 2>&1 || (curl -fsSL https://ollama.com/install.sh | sh)

# 2. Start Ollama server in background
!pkill -f "ollama serve" || true
!nohup ollama serve > /tmp/ollama_colab.log 2>&1 &

# 3. Verify server health
print("Waiting for Ollama engine to initialize...")
for attempt in range(30):
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as resp:
            if resp.status == 200:
                print("✓ Ollama server online at http://localhost:11434")
                break
    except Exception:
        time.sleep(1)

### Step 2: Pull Both Required Models
Pulls the Primary LLM (`qwen2.5:7b-instruct`) and the Embedding model (`all-minilm`).

In [ ]:
print("Pulling Primary LLM: qwen2.5:7b-instruct (takes ~1-2 min on Colab GPU)...")
!ollama pull qwen2.5:7b-instruct

print("Pulling Embedding Model: all-minilm...")
!ollama pull all-minilm

print("\nVerified Models in Ollama:")
!ollama list

### Step 3: Install Python Dependencies

In [ ]:
!pip install -q pydantic==2.8.2 pydantic-settings==2.3.4 neo4j==5.23.0 httpx==0.27.0 numpy==1.26.4 orjson==3.10.7 async-lru==2.0.4 tenacity==8.5.0 python-dotenv==1.0.1 nltk>=3.8.1 pandas
import nltk
nltk.download("punkt", quiet=True)
print("✓ All Python dependencies ready!")

### Step 4: Configure Neo4j Cloud AuraDB Credentials & Environment
Connects to Neo4j Cloud AuraDB using your credentials. If the AuraDB instance is dormant or paused, M4 automatically engages the built-in High-Speed In-Memory Graph Store so testing proceeds seamlessly!

In [ ]:
import os

# Neo4j Cloud AuraDB Credentials
os.environ["NEO4J_URI"] = "neo4j+s://8d2d5bc7.databases.neo4j.io"
os.environ["NEO4J_USER"] = "8d2d5bc7"
os.environ["NEO4J_PASSWORD"] = "bw8o9R3NXXApN2DXy72Gv7iXNI2KcVDGp_zZ7h0D5BU"

# Engineering Optimizations (Doc Page 4)
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "3"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"

print("✓ Environment Configured:")
print(f"  • Neo4j Target:    {os.environ['NEO4J_URI']}")
print(f"  • Neo4j User:      {os.environ['NEO4J_USER']}")
print(f"  • Ollama Base URL: {os.environ['OLLAMA_BASE_URL']}")

### Step 5: Run LoCoMo Benchmark Evaluation Suite
Ingests dialogue sessions and evaluates questions across all 4 categories:
- **Category 1**: Single-Hop Reasoning
- **Category 2**: Temporal Reasoning
- **Category 3**: Multi-Hop Reasoning
- **Category 4**: Open Domain Reasoning

In [ ]:
# Execute evaluation suite
!python run_locomo_eval.py --samples-per-cat 3 --max-sessions 4

### Step 6: Render Academic Comparison Table
Displays benchmark comparison table matching Section 6 of the Project Presentation and Paper Table III.

In [ ]:
import json
import pandas as pd

try:
    with open("locomo_evaluation_results.json", "r") as f:
        eval_data = json.load(f)
    
    s = eval_data.get("summary", {})
    rows = [
        {"Framework / Method": "MemGPT [33]", "Multi-Hop (F1)": "10.44%", "Temporal (F1)": "4.21%", "Open Dom (F1)": "13.42%", "Single-Hop (F1)": "9.56%", "Context Tokens": "~16,953"},
        {"Framework / Method": "Mem0 [35]", "Multi-Hop (F1)": "17.81%", "Temporal (F1)": "12.33%", "Open Dom (F1)": "12.33%", "Single-Hop (F1)": "29.04%", "Context Tokens": "~1,484"},
        {"Framework / Method": "A-Mem [36]", "Multi-Hop (F1)": "18.23%", "Temporal (F1)": "24.32%", "Open Dom (F1)": "16.48%", "Single-Hop (F1)": "23.63%", "Context Tokens": "~1,300"},
        {"Framework / Method": "M⁴ Paper Table III", "Multi-Hop (F1)": "25.05%", "Temporal (F1)": "26.02%", "Open Dom (F1)": "16.70%", "Single-Hop (F1)": "25.61%", "Context Tokens": "~2,474"},
        {"Framework / Method": "M⁴ (Our Test Run)", "Multi-Hop (F1)": f"{s.get('multihop_f1', 0.0):.2f}%", "Temporal (F1)": f"{s.get('temporal_f1', 0.0):.2f}%", "Open Dom (F1)": f"{s.get('opendomain_f1', 0.0):.2f}%", "Single-Hop (F1)": f"{s.get('singlehop_f1', 0.0):.2f}%", "Context Tokens": f"~{s.get('avg_context_tokens', 0)}"},
    ]
    df = pd.DataFrame(rows)
    display(df)
except Exception as e:
    print(f"Note: {e}")

### Step 7: Launch Interactive Mentor Demonstration UI (Directly in Colab!)
Starts the interactive Web UI and displays it directly inside your Colab notebook window or via a free public link.

In [ ]:
# 1. Launch UI and Cloudflare Tunnel
!bash colab_setup_and_launch_all.sh

# 2. Display UI directly inside Colab Notebook!
import time
from IPython.display import display, HTML
try:
    from google.colab import output
    print("Rendering live UI directly inside your Colab notebook:")
    output.serve_kernel_port_as_iframe(7860, height=850)
except Exception:
    print("UI running on port 7860. Access via the public tunnel link printed above!")
